# 17 · Import Official Cisco Datasheets and SEC Proxy Documents

**Run All in VS Code.** Additive update: preserves your existing 22 PDFs, research/analytics data, Market Basket implementation, and SQLite index. The seven uploaded original PDFs are included under `data/course_sources/official_pdfs/`.

**Classification is verified from each document's cover:** four Cisco technical datasheets go to `technology`; Agilent 2025/2026 DEF 14A proxy statements and KLA 2024 DEF 14A go to `sec`. **None of these SEC documents is a 2025 Form 10-K.** A proxy discusses governance and voting; it is not a substitute for 10-K financial-risk disclosures. Source links and SHA-256 hashes are recorded. Rerunning is safe because identical file contents are deduplicated.

The `clinical` Agilent equipment-manual gap is **not** filled by the Agilent *proxy* statements.

In [1]:
%run 01_document_ingestion.ipynb
import json, hashlib
from pathlib import Path
from pprint import pprint
PDF_DIR=ROOT/'data'/'course_sources'/'official_pdfs'
SOURCE_MANIFEST=PDF_DIR/'official_document_manifest.json'
assert SOURCE_MANIFEST.exists(), f'Missing document manifest: {SOURCE_MANIFEST}'
official_docs=json.loads(SOURCE_MANIFEST.read_text())
print('Found',len(official_docs),'official PDF records')

Database initialized: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Document ingestion functions ready
Found 7 official PDF records


In [2]:
# Verify file contents, provenance, and category BEFORE they enter the RAG index.
import fitz
from collections import Counter

def verify_official_file(row):
    path=PDF_DIR/row['filename']
    if not path.is_file():raise FileNotFoundError(f'Missing {path}')
    raw=path.read_bytes()
    if hashlib.sha256(raw).hexdigest()!=row['sha256']:
        raise ValueError(f'Content checksum mismatch: {row["filename"]}')
    if not raw.startswith(b'%PDF-'):
        raise ValueError(f'File is not a PDF: {row["filename"]}')
    with fitz.open(stream=raw,filetype='pdf') as pdf:
        if not pdf.page_count:raise ValueError('PDF has no pages')
        # Validate the actual document text, NOT filename-only assertions.
        head=' '.join((pdf[p].get_text('text') or '') for p in range(min(3,pdf.page_count))).casefold()
        if row['document_type']=='sec_def14a_proxy':
            if 'schedule 14a' not in head and 'proxy statement' not in head:
                raise ValueError('Incorrect SEC classification: '+row['filename'])
            if row['organization']=='KLA Corporation' and 'kla corporation' not in head:
                raise ValueError('KLA identity mismatch')
            if row['organization']=='Agilent Technologies' and 'agilent technologies' not in head:
                raise ValueError('Agilent identity mismatch')
        elif row['document_type']=='technical_datasheet':
            if 'cisco' not in head or 'data sheet' not in head:
                raise ValueError('Incorrect Cisco datasheet classification')
        else:raise ValueError('Unsupported document type')
        return pdf.page_count

results=[]
for meta in official_docs:
    page_count=verify_official_file(meta)
    # source_type is a factual category, not a claim that this is a 10-K or a certified part.
    source_type='official-cisco-datasheet' if meta['domain']=='technology' else 'sec-def14a-proxy'
    row=ingest_blob(PROJECT_ID,meta['domain'],meta['filename'],(PDF_DIR/meta['filename']).read_bytes(),
                    source_type=source_type,source_url=meta['source_url'])
    results.append({**{key:meta[key] for key in ['filename','organization','domain','document_type','year']},
                    'pages':page_count,**row})

report={'documents_processed':len(results),'newly_indexed':sum(not r['duplicate'] for r in results),
        'new_chunks':sum(r.get('chunks',0) for r in results),
        'already_indexed':sum(r['duplicate'] for r in results),
        'by_domain':dict(Counter(r['domain'] for r in results)),
        'file_details':results,
        'notice':'Agilent SEC filings here are DEF 14A proxies; 2025 10-K is still not supplied.'}
print(json.dumps({k:v for k,v in report.items() if k!='file_details'},indent=2))
for r in results:
    print(f"{r['domain']:11s} {r['filename']:36s} {r['pages']:3d} pp  "
          f"{'already indexed' if r['duplicate'] else str(r['chunks'])+' new chunks'}  "
          f"{r['organization']} ({r['document_type']})")
assert len(results)==7 and all(r['status']=='ready' for r in results)

{
  "documents_processed": 7,
  "newly_indexed": 7,
  "new_chunks": 1346,
  "already_indexed": 0,
  "by_domain": {
    "technology": 4,
    "sec": 3
  },
  "notice": "Agilent SEC filings here are DEF 14A proxies; 2025 10-K is still not supplied."
}
technology  datasheet-c78-744088.pdf              24 pp  58 new chunks  Cisco (technical_datasheet)
technology  data_sheet-c78-732542.pdf             28 pp  81 new chunks  Cisco (technical_datasheet)
technology  datasheet-c78-731632.pdf              31 pp  91 new chunks  Cisco (technical_datasheet)
technology  catalyst-8000v-edge-software-ds.pdf   19 pp  47 new chunks  Cisco (technical_datasheet)
sec         agilent.pdf                           95 pp  351 new chunks  Agilent Technologies (sec_def14a_proxy)
sec         agient2.pdf                           87 pp  341 new chunks  Agilent Technologies (sec_def14a_proxy)
sec         agilent3.pdf                          92 pp  377 new chunks  KLA Corporation (sec_def14a_proxy)


In [3]:
from pathlib import Path

base = ROOT / "data" / "course_sources" / "official_pdfs"

folders = {
    "Clinical": "clinical",
    "FoodSecurity": "csr",
    "UnitedStatesSecuritiesandExchangeCommission(SEC)": "sec",
    "Technology": "technology",
}

results = []
failures = []

for folder_name, domain in folders.items():
    folder = base / folder_name

    if not folder.exists():
        print("Folder not found:", folder_name)
        continue

    for pdf in sorted(folder.rglob("*.pdf")):
        try:
            result = ingest_file(
                pdf,
                domain,
                project_id=PROJECT_ID,
                source_type="course-provided-pdf"
            )

            results.append(result)

            status = (
                "ALREADY INDEXED"
                if result.get("duplicate")
                else "NEWLY INDEXED"
            )

            print(status, "|", domain, "|", pdf.name)

        except Exception as error:
            failures.append((str(pdf), str(error)))
            print("FAILED:", pdf.name, error)

print("\nIMPORT COMPLETE")
print("PDFs processed:", len(results))
print("New PDFs:", sum(not r.get("duplicate", False) for r in results))
print("Already indexed:", sum(r.get("duplicate", False) for r in results))
print("Failures:", len(failures))

NEWLY INDEXED | clinical | Analysis_and_Prediction_of_COVID-19_using_Machine_Learning.pdf
NEWLY INDEXED | clinical | Big_Data_Analytics_on_Irritating_Anxiety_of_Long_COVID.pdf
NEWLY INDEXED | clinical | COVID-19_Update_Forecast_and_Assistant_-_An_Interactive_Web_Portal_to_Provide_Real-Time_Information_and_Forecast_COVID-19_Cases_in_Bangladesh.pdf
NEWLY INDEXED | clinical | COVID_Detection_using_Deep_Learning.pdf
NEWLY INDEXED | clinical | Extraction_of_Features_from_Lung_Image_for_the_Detection_of_Covid-19.pdf
NEWLY INDEXED | clinical | Artificial_Intelligence_and_Machine_Learning_for_Improving_Glycemic_Control_in_Diabetes_Best_Practices_Pitfalls_and_Opportunities.pdf
NEWLY INDEXED | clinical | Deep_Learning-Based_Genetic_Detection_and_Pathogenesis_of_Diabetes.pdf
NEWLY INDEXED | clinical | Exploring_Biomarker_Relationships_in_Both_Type_1_and_Type_2_Diabetes_Mellitus_Through_a_Bayesian_Network_Analysis_Approach.pdf
NEWLY INDEXED | clinical | Improving_Self-Management_of_Type_2_Diabetes

### What these documents can and cannot prove

- **Cisco:** The four product datasheets support product specifications, interfaces, module families, performance, and other published capabilities. A generic datasheet does **not** prove which individual part failed or that any candidate SKU works with the user's precise hardware/software configuration.
- **SEC:** Agilent 2025 and 2026 plus KLA 2024 are **DEF 14A proxy** documents. Useful for board, governance, shareholder-voting, and certain compensation disclosures. **Do not answer 2025 annual Form 10-K financial risk questions as if these were 10-Ks.**
- **Clinical:** Agilent proxy filings should never be routed as laboratory equipment or diagnostic-device manuals.
- **Original RAG:** The offline search is TF-IDF/BM25; embedding and native pgvector connectors are separate optional integrations.

Next: **Run All in `18_official_pdf_rag_checks.ipynb`** to verify citations, provenance, and missing-10-K guardrails.